# Fraud Detection — End-to-End MLOps on AWS

A **portfolio** project that demonstrates a realistic and **reproducible** MLOps flow:
train a **fraud detection** model (Fraud / Not Fraud), evaluate it with a **configurable**
metric, register it in the **Model Registry** only if it clears the threshold, deploy it to a
**real-time endpoint** and let **the arrival of new data trigger the pipeline** again.

```text
Synthetic dataset (sklearn) -> S3 Raw -> Processing -> Training -> Evaluation
  -> Conditional (metric >= threshold) -> Model Registry -> Approval -> Endpoint -> Inference
```

* **SageMaker Pipelines SDK V3** (the same patterns validated in this environment).
* This is the **thin** notebook: all the logic lives in [`src/fraud_mlops/`](src/fraud_mlops/)
  and the notebook only orchestrates the E2E demo in 14 sections.
* The original 53-cell notebook used to build the project is kept as a backup in
  `notebooks/archive/` (local; not published, see `.gitignore`).


## ONE-TIME MANUAL SETUP

* **AWS credentials** already configured on your machine (default profile).
* **Execution role** for SageMaker with full permissions (pipelines, processing, training,
  Model Registry, deployment and invoke). The notebook auto-detects `get_execution_role()`; if
  it fails, set the `SAGEMAKER_ROLE_ARN` environment variable before running.
* **Optional environment variables** (no code changes):

  | Variable | Purpose |
  |---|---|
  | `AWS_REGION` | AWS region (default `us-east-1`) |
  | `SAGEMAKER_ROLE_ARN` | Role ARN if auto-detection is not possible |
  | `SNS_ALERT_EMAIL` | Email to subscribe to the SNS alert topic |

* **S3 bucket**: the default SageMaker bucket is used (`sagemaker-<region>-<account>`).
* **Quota** `ml.m5.large`: processing + training + endpoint.

> Navigate to the project root and run the cells **in order**.
> Always read the **COST WARNING** notes before each step that consumes AWS.


## 1. Configuration

Every knob of the project lives in [`src/fraud_mlops/config.py`](src/fraud_mlops/config.py)
and is imported with a single line. They are **default** values: most of them can be overridden
on each **pipeline run** (parameters), as you will see in section 10.


In [ ]:
import sys
sys.path.insert(0, "src")   # the fraud_mlops package lives in src/

import json
import pandas as pd
from IPython.display import display

from dotenv import load_dotenv
load_dotenv()

from fraud_mlops import config

print("Pipeline:", config.PIPELINE_NAME, "| Region:", config.AWS_REGION)
print("Default metric:", config.METRIC_NAME, ">=", config.METRIC_THRESHOLD)
print("Features:", len(config.FEATURE_COLS), "| Code dir:", config.CODE_DIR)

## 2. Sessions

`get_sessions()` creates role, bucket and every boto3 client **once**. Key ideas of SDK V3:

* **`Session()`** → runs *now* (upload data, deploy, queries, cleanup).
* **`PipelineSession()`** → *captures* the pipeline intent; the service runs it on launch.


In [ ]:
from fraud_mlops.session import get_sessions
from fraud_mlops import events

sessions = get_sessions()
print("Region:", sessions.region)
print("Bucket:", sessions.bucket)
print("Prefix:", sessions.prefix)

# S3 -> EventBridge enabled on the bucket (idempotent, one time only)
events.enable_s3_eventbridge(sessions)


## 3. Dataset

We generate a **synthetic** fraud dataset (imbalanced binary problem) with
`make_classification`, in the style of **Kaggle Credit-Card Fraud**: 20 numeric features
(`time`, `amount` and 18 `vXX` components). With 5,000 rows it is small, fast and cheap.


In [ ]:
from fraud_mlops import dataset

df = dataset.make_fraud_dataframe(seed=42)

print("Dimensions:", df.shape)
print(df[config.TARGET_COL].value_counts(normalize=True).rename("proportion"))
dataset.plot_overview(df)


## 4. S3

We upload the **raw** CSV to the *landing zone* `s3://…/<prefix>/ingest/raw.csv`. That prefix
is the one **EventBridge** will watch to trigger the pipeline when new data arrives.


In [ ]:
raw_data_s3 = dataset.upload_dataframe(sessions, df)
probe_df = dataset.sample_probe(df)      # 5 sample rows for section 13

print("Raw data in S3:", raw_data_s3)
print("Sample rows kept in memory:", probe_df.shape)


## 5. Data Processing

A `ScriptProcessor` with the **Scikit-learn** container:

1. Reads the raw CSV from S3 (the path comes from the **pipeline parameter** `InputData`).
2. Splits into `train / validation / test` (target first, no header → XGBoost CSV format).
3. Trains a **simple baseline** (`LogisticRegression`) and writes `baseline.json`.

The **baseline** is the reference model: the pipeline compares (outside the DAG) the main
model (XGBoost) against it.


## 6. Training

The main model is **XGBoost** (SageMaker-managed container) trained with `ModelTrainer`. The
metric is configurable, but by default we use `roc_auc` because it is more informative than
accuracy on imbalanced data (30% fraud).


## 7. Evaluation

The evaluation `ScriptProcessor` loads the training `model.tar.gz`, evaluates it with the
classification metrics and writes `evaluation.json`. The **ConditionStep** cannot change the
JSON path dynamically: that is why [`code/evaluate.py`](code/evaluate.py) receives `--metric`
(a **pipeline parameter**) and stores the chosen metric under the **fixed** key
`binary_classification_metrics.metric.value`. So changing `MetricName` on every run really
changes the criterion that opens the PASS/FAIL branch.


## 8. Conditional Logic

The **ConditionStep** decides with the configured metric:

```text
IF  metric (JsonGet) >= threshold (parameter)  -> PASS  -> Model Registry
ELSE                                           -> FAIL  -> FailStep
                                                            -> run in Failed state
                                                            -> EventBridge (fraud-pipeline-failed-alert rule)
                                                            -> SNS -> email
```

* `metric_threshold` is a **pipeline parameter**: you can change it on every run.
* On FAIL nothing is registered and nothing is deployed.
* The FailStep does **not** send the email: it only moves the run to `Failed`.
  The notification comes from an **EventBridge rule** that watches that state (section 11).


## 9. Model Registry

Only if `metric_value >= metric_threshold` the model is registered:

* `ModelMetrics` attaches `evaluation.json` as model metadata (human review).
* `ModelStep` + `ModelBuilder.register(...)` create the registration step in the DAG.
* `approval_status` is a parameter: by default **`PendingManualApproval`** (human review).
  If you pass `ModelApprovalStatus = Approved`, the run ends up approved automatically.

We create the registration group if it does not exist (idempotent):


In [ ]:
from fraud_mlops import deploy

deploy.ensure_model_package_group(sessions)


## 10. Pipeline

`build_pipeline()` joins every step in the DAG (definition in
[`src/fraud_mlops/pipeline.py`](src/fraud_mlops/pipeline.py)). Note: **the pipeline does NOT
deploy by itself**. The *integration* with production happens in the Model Registry and
**the approval triggers the deploy** in section 12 (standard pattern: *deploy never deploys
what is not approved*).


In [ ]:
from fraud_mlops.pipeline import build_pipeline, preview_definition

pipeline = build_pipeline(sessions, input_data_s3=raw_data_s3)

preview = preview_definition(pipeline)     # only serializes; does NOT run the pipeline
print("Pipeline:", pipeline.name)
print("Steps:", [s["Name"] for s in preview["Steps"]])
print("Parameters:", [p["Name"] for p in preview["Parameters"]])


### ⚠️ COST WARNING

This section **launches the pipeline on AWS** (creates/uses resources that incur cost):

* **What is created:** Processing jobs (split + eval) and Training (XGBoost `ml.m5.large`),
  plus a version in the Model Registry. ~10-15 min per run. Rough cost: **cents** per run.
* **How to minimize it:** small instances (`ml.m5.large`) and the **pipeline caching** on
  `SplitData`/`EvaluateFraud` avoid re-running steps when the input has not changed.
* **How to remove it:** section **14. Cleanup**.


In [ ]:
from fraud_mlops import runner

# Run parameters. Change them here to try other configurations:
#   - ModelApprovalStatus="Approved"  -> auto-approves (no human step)
#   - MetricName="f1", MetricThreshold=0.55 -> a different evaluation criterion
run_params = config.build_run_params(raw_data_s3)
execution_arn = runner.upsert_and_start(sessions, pipeline, run_params)


In [ ]:
status = runner.wait_execution(sessions, execution_arn)
runner.report_steps(sessions, execution_arn)

print("Final status:", status)
if status != "Succeeded":
    print("Check FailureReason in the SageMaker Pipelines console.")


In [ ]:
if status == "Succeeded":
    comparison = runner.compare_with_baseline(sessions, threshold=config.METRIC_THRESHOLD)
    display(comparison)
else:
    print("The pipeline did not reach Succeeded; check the console before continuing.")


## 11. Event Trigger and Alerts

Complete event architecture (no Lambdas):

```text
S3 (ObjectCreated on ingest/) -> EventBridge -> SageMaker Pipeline   [rule 1]
Pipeline Executing (start)    -> EventBridge -> SNS -> email        [rule 2]
Pipeline Failed  (failure)    -> EventBridge -> SNS -> email        [rule 3]
```

* **Rule 1** `fraud-new-data-trigger`: `ObjectCreated` on `ingest/` → native SageMaker target
  with `SageMakerPipelineParameters`.
* **Rule 2** `fraud-pipeline-started-alert`: `detail.currentPipelineExecutionStatus = Executing`.
* **Rule 3** `fraud-pipeline-failed-alert`: `detail.currentPipelineExecutionStatus = Failed`.
* The alert rules filter on `detail.pipelineArn` (the ARN prefix of this pipeline).
  That is the **real** event field: `PipelineName` / `PipelineExecutionStatus` **do not exist**
  in the payload and a pattern using them never matches (the original failure of this demo).
* The SNS topic needs `events.amazonaws.com -> sns:Publish` in its **policy**
  (`ensure_sns_topic` adds it). Without that permission EventBridge cannot publish and
  **nothing arrives**, even though the rule does fire.
* The email subscription must be **confirmed** from the SNS welcome email.


In [ ]:
events_role_arn = events.ensure_events_role(sessions)
topic_arn = events.ensure_sns_topic(sessions)
events.ensure_event_rules(sessions, events_role_arn, topic_arn, input_data_s3=raw_data_s3)

# Cheap verification (runs no pipeline and sends no emails):
#   1) the patterns match the real event documented by SageMaker
#   2) the EventBridge -> SNS -> email chain is healthy
events.check_event_patterns(sessions)
events.diagnose_alerts(sessions, topic_arn)


### ⚠️ COST WARNING — Trigger demonstration

Uploading a new CSV will trigger **a full pipeline run** (~10-15 min, `ml.m5.large`).

Optional: you can skip this step and run section 12 with the already registered version.


In [ ]:
# We simulate a new data delivery: another synthetic CSV in the same landing zone.
df_nuevo = dataset.make_fraud_dataframe(seed=999)
print("df_nuevo:", df_nuevo.shape,
      "| target proportion:",
      df_nuevo[config.TARGET_COL].value_counts(normalize=True).round(3).to_dict())

dataset.upload_dataframe(sessions, df_nuevo)   # same key -> EventBridge triggers the pipeline
print("New CSV uploaded -> EventBridge should trigger the pipeline...")


In [ ]:
# Wait until the running execution shows up (triggered automatically)
auto_exec = events.wait_for_automatic_execution(sessions)


In [ ]:
# (Optional) Wait for that run and publish a test message to SNS
if auto_exec and auto_exec["PipelineExecutionStatus"] == "Executing":
    status_auto = runner.wait_execution(sessions, auto_exec["PipelineExecutionArn"])
    print("Final status of the EventBridge-triggered run:", status_auto)
else:
    status_auto = "no disponible"

if topic_arn:
    events.publish_alert(sessions, topic_arn,
                         subject="Fraud MLOps",
                         message=f"Ejecucion '{auto_exec}' -> {status_auto}")


### ⚠️ COST WARNING — (Optional) Demonstrate the FAIL branch + alert

With an impossible threshold (0.99) the PASS never happens: the run ends in **Failed** and the
`fraud-pipeline-failed-alert` rule publishes to SNS → failure email. Since every run goes
through `Executing` first, you will also get the **start** email
(`fraud-pipeline-started-alert`). It generates one extra run (or only a few seconds if the
steps hit the cache).


In [ ]:
run_fail_arn = runner.upsert_and_start(
    sessions, pipeline, config.build_run_params(raw_data_s3, MetricThreshold=0.99))

status_fail = runner.wait_execution(sessions, run_fail_arn)
runner.report_steps(sessions, run_fail_arn)

print("Final status (expected Failed):", status_fail)
assert status_fail == "Failed", f"Expected Failed but got {status_fail}"


### Alert verification (read only)

We check with CloudWatch metrics whether the rules **fired** on that run
(`TriggeredRules`) and whether SNS **published** the emails. Metrics take ~1 min to appear.


In [ ]:
report = events.diagnose_alerts(sessions, topic_arn, hours=1)
report["metrics"]


## 12. Deployment

Golden rule of the project: **what is not approved does not get deployed**.

1. List the Model Registry versions and their approval status.
2. If `ApprovalStatus != Approved`, **block the deploy** (gate: manual/configured).
3. Only with `Approved` we create `Model` → `EndpointConfig` → **Real-Time Endpoint** (`ml.m5.large`).

In production the approval comes from a human (console) or an external process (Lambda/CI). Here
we demonstrate it with step 2 and keep the deploy conditioned on that real state.


In [ ]:
version = deploy.latest_model_version(sessions)
latest_arn = version["ModelPackageArn"]

deploy.ensure_approved(sessions, latest_arn)

# Idempotent: if the endpoint already exists, it does NOT deploy again
endpoint, created = deploy.ensure_endpoint(sessions, latest_arn)
print("Endpoint created in this session:", created)


## 13. Inference

Two ways to invoke the endpoint (both from this notebook, in Python):

1. **From Python (boto3)** → `sagemaker-runtime.invoke_endpoint`.
2. **From the notebook (SDK)** → the `Endpoint` object returned by `deploy` (if it was just created).

XGBoost (`text/csv`, no header) returns **one probability per line**; the class is
`prob >= 0.5`.


In [ ]:
from fraud_mlops import inference

preds = inference.predict_dataframe(sessions, probe_df, endpoint=endpoint)
display(preds[[*config.FEATURE_COLS, config.TARGET_COL, "probability", "prediction"]])


In [ ]:
row = probe_df.iloc[2]
prob = inference.predict_row(sessions, row, endpoint=endpoint)

print("Probability:", round(prob, 4))
print("Class    :", "FRAUD" if prob >= 0.5 else "NOT FRAUD",
      f"(true label: {'FRAUD' if row[config.TARGET_COL] == 1 else 'NOT FRAUD'})")


## 14. Cleanup

**Essential when you finish the lab** so you do not keep paying:

* Endpoint + EndpointConfig + Model.
* Pipeline (definition).
* Project Model Registry groups (versions included, with retries).
* EventBridge rules + events role + SNS topic.
* **S3 → EventBridge** notification on the bucket (disabled by the cleanup itself).
* (Optional, warned) S3 data under the project prefix.

> ⚠️ **If you touched `src/` during the session** (e.g. `config.py`), the kernel is still using
> the copy it imported at the beginning: **restart the kernel** and re-run the configuration
> cell (and any you need) **before** this one. Otherwise you will get
> `AttributeError: module 'fraud_mlops.config' has no attribute ...`.


### ⚠️ COST WARNING

Deleting the **endpoint** stops the live instance billing (the biggest expense of the project).


In [ ]:
from fraud_mlops import cleanup

cleanup.cleanup_resources(sessions, pipeline=pipeline, topic_arn=topic_arn)
cleanup.cleanup_s3(sessions, delete=False)   # True -> deletes the whole project prefix


## Key Takeaways

* **E2E pipeline**: Split → Train → Evaluate → Conditional → Register. It does NOT deploy by itself.
* The **metric and threshold** are pipeline parameters (the metric is chosen in `MetricName`).
* **Conditional**: `metric >= threshold` → registers; otherwise → `FailStep` → run `Failed`.
* **Alerts**: EventBridge sends the email, not the FailStep. There are **3 rules**:
  new data → launches the pipeline; `Executing` → start email; `Failed` → failure email.
  Requirements: a pattern with `detail.currentPipelineExecutionStatus` + `detail.pipelineArn`,
  an SNS policy with `events.amazonaws.com` and a confirmed email subscription.
* **Model Registry**: `PendingManualApproval` by default; deploy is blocked until approved.
* **Event Trigger**: S3 (`ObjectCreated` on `ingest/`) → EventBridge → just launches the pipeline.
* **Cost**: small instances + caching + delete the endpoint when you are done.
* **Reusable code**: the logic lives in `src/fraud_mlops/`; this notebook only orchestrates
  the demo and documents each step.
